In [1]:
import os

print("Current directory:", os.getcwd())
print("Files:", os.listdir()[:30])

Current directory: /content
Files: ['.config', 'sample_data']


In [2]:
!python -c "import torch; print('PyTorch:', torch.__version__); print('CUDA available:', torch.cuda.is_available())"

PyTorch: 2.11.0+cu130
CUDA available: True


In [3]:
import sys
import subprocess

print("Notebook Python:", sys.version)

result = subprocess.run(
    [
        sys.executable,
        "-c",
        "import torch; print(torch.__version__); print(torch.cuda.is_available())"
    ],
    capture_output=True,
    text=True
)

print("Subprocess output:", result.stdout)
print("Subprocess errors:", result.stderr)

Notebook Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Subprocess output: 2.11.0+cu130
True

Subprocess errors: 


In [4]:
# ===== CONFIG =====
SAVE_DIR   = "."
EPOCHS     = 25
BATCH_SIZE = 256
SEED       = 42

import os, math, copy, time, random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from sklearn.metrics import classification_report, confusion_matrix

os.makedirs(SAVE_DIR, exist_ok=True)

def set_seed(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__, "| Device:", device)

PyTorch: 2.11.0+cu130 | Device: cuda


In [5]:
train_raw = torchvision.datasets.EMNIST("./data", split="byclass", train=True,  download=True)
test_raw  = torchvision.datasets.EMNIST("./data", split="byclass", train=False, download=True)

raw_classes = train_raw.classes

X_train_raw = train_raw.data.transpose(1, 2).contiguous()
y_train_raw = train_raw.targets.clone()
X_test_raw  = test_raw.data.transpose(1, 2).contiguous()
y_test_raw  = test_raw.targets.clone()

print("Train:", tuple(X_train_raw.shape), "| Test:", tuple(X_test_raw.shape))
print("Raw classes:", len(raw_classes))

100%|██████████| 562M/562M [00:03<00:00, 155MB/s]


Train: (697932, 28, 28) | Test: (116323, 28, 28)
Raw classes: 62


In [6]:
# Only merge same letter upper/lower case. No digit-letter merges.
LOWER_LOOKALIKES = "cfilmnoprsuvwxyz"
SHAPE_MERGES = {}

def resolve(ch):
    if ch in LOWER_LOOKALIKES:
        ch = ch.upper()
    return SHAPE_MERGES.get(ch, ch)

groups = {}
for ch in raw_classes:
    groups.setdefault(resolve(ch), []).append(ch)

final_keys    = list(groups.keys())
final_classes = ["/".join(groups[k]) for k in final_keys]
NUM_CLASSES   = len(final_classes)

label_map = torch.tensor([final_keys.index(resolve(ch)) for ch in raw_classes])

print("Classes:", NUM_CLASSES)
print(final_classes)

Classes: 46
['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', 'A', 'B', 'C/c', 'D', 'E', 'F/f', 'G', 'H', 'I/i', 'J', 'K', 'L/l', 'M/m', 'N/n', 'O/o', 'P/p', 'Q', 'R/r', 'S/s', 'T', 'U/u', 'V/v', 'W/w', 'X/x', 'Y/y', 'Z/z', 'a', 'b', 'd', 'e', 'g', 'h', 'j', 'k', 'q', 't']


In [7]:
y_train_all = label_map[y_train_raw]
y_test_all  = label_map[y_test_raw]

g = torch.Generator().manual_seed(SEED)
perm = torch.randperm(len(X_train_raw), generator=g)
n_val = int(0.1 * len(perm))
val_idx, tr_idx = perm[:n_val], perm[n_val:]

X_train = X_train_raw[tr_idx].to(device)
y_train = y_train_all[tr_idx].to(device)
X_val   = X_train_raw[val_idx].to(device)
y_val   = y_train_all[val_idx].to(device)
X_test  = X_test_raw.to(device)
y_test  = y_test_all.to(device)

print("Train:", len(X_train), "| Val:", len(X_val), "| Test:", len(X_test))

Train: 628139 | Val: 69793 | Test: 116323


In [8]:
def conv_block(cin, cout):
    return nn.Sequential(
        nn.Conv2d(cin,  cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
        nn.MaxPool2d(2),
    )

class CharacterCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            conv_block(1, 64),
            conv_block(64, 128),
            conv_block(128, 256),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(256 * 3 * 3, 512), nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        return self.head(self.features(x))

model = CharacterCNN(NUM_CLASSES).to(device)
print("Parameters:", sum(p.numel() for p in model.parameters()))

Parameters: 2348910


In [9]:
def batches(X, y, bs, shuffle):
    idx = torch.randperm(len(X), device=X.device) if shuffle else torch.arange(len(X), device=X.device)
    for i in range(0, len(X), bs):
        j = idx[i:i + bs]
        yield X[j].float().div(255).unsqueeze(1), y[j]

def augment(x, max_rot=15, max_scale=0.15, max_shift=0.15):
    B, dev = x.size(0), x.device
    r = lambda: torch.rand(B, device=dev) * 2 - 1
    ang = r() * math.radians(max_rot)
    sc  = 1 + r() * max_scale
    cos, sin = torch.cos(ang) / sc, torch.sin(ang) / sc
    theta = torch.stack([
        torch.stack([cos, -sin, r() * max_shift], dim=1),
        torch.stack([sin,  cos, r() * max_shift], dim=1),
    ], dim=1)
    grid = F.affine_grid(theta, x.shape, align_corners=False)
    return F.grid_sample(x, grid, padding_mode="zeros", align_corners=False)

@torch.no_grad()
def evaluate(X, y, bs=1024):
    model.eval()
    preds, total_loss = [], 0.0
    for xb, yb in batches(X, y, bs, shuffle=False):
        out = model(xb)
        total_loss += F.cross_entropy(out, yb, reduction="sum").item()
        preds.append(out.argmax(1))
    preds = torch.cat(preds)
    acc = (preds == y).float().mean().item() * 100
    return acc, total_loss / len(X), preds.cpu().numpy()

In [10]:
model = CharacterCNN(NUM_CLASSES).to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
steps_per_epoch = math.ceil(len(X_train) / BATCH_SIZE)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=3e-3, epochs=EPOCHS, steps_per_epoch=steps_per_epoch
)
use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

CKPT_PATH = os.path.join(SAVE_DIR, "emnist_cnn_case_merged_best.pth")
history = {"train_loss": [], "val_loss": [], "val_acc": []}
best_acc, best_state = 0.0, None

for epoch in range(EPOCHS):
    t0 = time.time()
    model.train()
    running = 0.0
    for xb, yb in batches(X_train, y_train, BATCH_SIZE, shuffle=True):
        xb = augment(xb)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, enabled=use_amp):
            loss = criterion(model(xb), yb)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        running += loss.item()

    val_acc, val_loss, _ = evaluate(X_val, y_val)
    history["train_loss"].append(running / steps_per_epoch)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    flag = ""
    if val_acc > best_acc:
        best_acc = val_acc
        best_state = copy.deepcopy(model.state_dict())
        torch.save({"model": best_state, "classes": final_classes, "val_acc": best_acc}, CKPT_PATH)
        flag = "  <- saved"
    print(f"Epoch {epoch+1:2d}/{EPOCHS} | train loss {history['train_loss'][-1]:.4f} | "
          f"val loss {val_loss:.4f} | val acc {val_acc:.2f}% | {time.time()-t0:.0f}s{flag}")

model.load_state_dict(best_state)
print(f"\nBest validation accuracy: {best_acc:.2f}%")

Epoch  1/25 | train loss 1.1780 | val loss 0.3924 | val acc 89.54% | 64s  <- saved
Epoch  2/25 | train loss 1.0079 | val loss 0.3671 | val acc 89.69% | 61s  <- saved
Epoch  3/25 | train loss 0.9897 | val loss 0.4013 | val acc 88.89% | 63s
Epoch  4/25 | train loss 0.9778 | val loss 0.3518 | val acc 90.02% | 64s  <- saved
Epoch  5/25 | train loss 0.9658 | val loss 0.3748 | val acc 89.11% | 65s
Epoch  6/25 | train loss 0.9591 | val loss 0.3344 | val acc 90.52% | 65s  <- saved
Epoch  7/25 | train loss 0.9533 | val loss 0.3601 | val acc 89.69% | 65s
Epoch  8/25 | train loss 0.9475 | val loss 0.3504 | val acc 90.26% | 65s
Epoch  9/25 | train loss 0.9411 | val loss 0.3330 | val acc 90.86% | 65s  <- saved
Epoch 10/25 | train loss 0.9366 | val loss 0.3519 | val acc 90.01% | 65s
Epoch 11/25 | train loss 0.9313 | val loss 0.3479 | val acc 89.97% | 65s
Epoch 12/25 | train loss 0.9271 | val loss 0.3439 | val acc 90.30% | 65s
Epoch 13/25 | train loss 0.9224 | val loss 0.3351 | val acc 90.50% | 65s
E

In [11]:
test_acc, test_loss, y_pred = evaluate(X_test, y_test)
y_true = y_test.cpu().numpy()

print(f"Test accuracy: {test_acc:.2f}% | Test loss: {test_loss:.4f}\n")
print(classification_report(y_true, y_pred, labels=list(range(NUM_CLASSES)),
                            target_names=final_classes, digits=3, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
off = cm.copy()
np.fill_diagonal(off, 0)
flat = np.argsort(-off.ravel())[:15]
print(f"\n{'true':>12}  ->  {'predicted':<12}  count")
for f in flat:
    t, p = np.unravel_index(f, off.shape)
    print(f"{final_classes[t]:>12}  ->  {final_classes[p]:<12}  {off[t, p]:5d}")

Test accuracy: 90.82% | Test loss: 0.3304

              precision    recall  f1-score   support

           0      0.721     0.717     0.719      5778
           1      0.645     0.961     0.772      6330
           2      0.974     0.960     0.967      5869
           3      0.995     0.997     0.996      5969
           4      0.965     0.989     0.977      5619
           5      0.959     0.954     0.956      5190
           6      0.986     0.973     0.979      5705
           7      0.987     0.997     0.992      6139
           8      0.991     0.989     0.990      5633
           9      0.918     0.988     0.952      5686
           A      0.963     0.990     0.976      1062
           B      0.938     0.983     0.960       648
         C/c      0.963     0.985     0.974      2171
           D      0.897     0.935     0.915       779
           E      0.967     0.989     0.978       851
         F/f      0.981     0.980     0.981      1840
           G      0.973     0.902     

In [18]:
import gradio as gr
import cv2
import numpy as np
from PIL import Image
import torch
import torch.nn.functional as F

# ------------------------------------------------------------------
# MODEL LOADING & CLASSES
# ------------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint = torch.load("emnist_cnn_case_merged_best.pth", map_location=device)

NUM_CLASSES = len(checkpoint["classes"])
model = CharacterCNN(NUM_CLASSES).to(device)
model.load_state_dict(checkpoint["model"])
model.eval()

CLASSES = checkpoint["classes"]

# ------------------------------------------------------------------
# PREPROCESS & PREDICT FUNCTIONS
# ------------------------------------------------------------------
def preprocess(gray: np.ndarray) -> np.ndarray | None:
    """
    Convert a grayscale drawing into the exact tensor / array
    that your model expects (28×28, float32, [0,1]).
    Return None on empty / invalid input.
    """
    if gray is None or gray.size == 0:
        return None

    img = cv2.resize(gray, (28, 28), interpolation=cv2.INTER_AREA)
    img = img.astype(np.float32) / 255.0
    return img

def predict(gray: np.ndarray, top_k: int = 5) -> dict[str, float]:
    """
    Run your trained model and return a dict {class_name: probability}.
    """
    x = preprocess(gray)
    if x is None:
        return {}

    # Convert to tensor, add batch and channel dimensions: shape (1, 1, 28, 28)
    tensor_x = torch.tensor(x, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(tensor_x)
        probs = F.softmax(logits, dim=1).squeeze(0)

    top_probs, top_indices = torch.topk(probs, k=top_k)

    preds = {}
    for prob, idx in zip(top_probs.cpu().numpy(), top_indices.cpu().numpy()):
        class_name = CLASSES[idx]
        preds[class_name] = float(prob)

    return preds

# ------------------------------------------------------------------
# GRADIO UI HELPERS
# ------------------------------------------------------------------
def to_gray(data):
    if data is None:
        return None
    img = data["composite"] if isinstance(data, dict) else data
    img = np.asarray(img)

    if img.ndim == 3:
        if img.shape[2] == 4 and img[..., 3].min() < 255:
            img = img[..., 3]
        else:
            img = img[..., :3].mean(axis=-1)

    if img.mean() > 127:
        img = 255 - img

    return img.astype(np.uint8)

def make_preprocess_preview(gray):
    """Return a PIL image of what the model actually sees (28×28)."""
    x = preprocess(gray)
    if x is None:
        return None

    x = np.squeeze(x)
    x = (x * 255).astype(np.uint8)
    x = cv2.resize(x, (140, 140), interpolation=cv2.INTER_NEAREST)
    return Image.fromarray(x)

def gradio_predict(data):
    if data is None:
        return None, {}, "Draw a character first"

    gray = to_gray(data)
    if gray is None or gray.mean() < 1:
        return None, {}, "Empty drawing"

    preds = predict(gray, top_k=5)
    if not preds:
        return None, {}, "Empty drawing"

    lines = []
    for i, (cls, prob) in enumerate(preds.items(), 1):
        bar = "█" * int(prob * 20)
        lines.append(f"{i}. {cls:<12} {prob*100:5.1f}%  {bar}")
    summary = "\n".join(lines)

    preview = make_preprocess_preview(gray)
    return preview, preds, summary

demo = gr.Interface(
    fn=gradio_predict,
    inputs=gr.Sketchpad(
        canvas_size=(280, 280),
        brush=gr.Brush(default_size=14, colors=["#000000"], color_mode="fixed"),
    ),
    outputs=[
        gr.Image(label="What the model sees (28×28)"),
        gr.Label(num_top_classes=5, label="Top predictions"),
        gr.Textbox(label="Clear ranking", lines=6),
    ],
    live=True,
    title="Handwritten Character Recognizer (90.8%)",
    description=(
        "Draw **one** character with a thick stroke.\n\n"
        "• Labels like **O/o**, **S/s**, **L/l** mean upper and lower case were merged.\n"
        "• When the model is unsure (e.g. between **0** and **O/o**), you will see both with similar percentages."
    ),
)

if __name__ == "__main__":
    demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://702326b1baa6217b8e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
